# Crash data - exploring the questions

Walks through the processed Crash parquet files (`FMCSA_data_processed/crash/`, built by `crash_build_parquet.py`)
and prints the DataFrames behind each data question / discrepancy. Data: crashes from 2015-01-01 onward.

In [1]:
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

DATA = "../FMCSA_data_processed/crash/"
RAW = "../FMCSA_data_raw/crash_parquet/"

## Load the tables

In [2]:
crash = pd.read_parquet(DATA + "crash.parquet")
vehicle = pd.read_parquet(DATA + "crash_vehicle.parquet")
carrier = pd.read_parquet(DATA + "crash_carrier.parquet")
event = pd.read_parquet(DATA + "crash_event.parquet")
dim_event = pd.read_parquet(DATA + "dim_event.parquet")
dim_weather = pd.read_parquet(DATA + "dim_weather.parquet")
dim_light = pd.read_parquet(DATA + "dim_light.parquet")
dim_config = pd.read_parquet(DATA + "dim_vehicle_config.parquet")

print("crash  ", crash.shape)
print("vehicle", vehicle.shape)
print("carrier", carrier.shape)
print("event  ", event.shape)
crash.head()

crash   (2026244, 38)
vehicle (2168212, 19)
carrier (2168212, 15)
event   (2684271, 6)


,crash_key,report_number,report_state,report_date,report_time,report_hour,state,state_fips,county_code,county_fips,county_fips_valid,city,location,agency,trafficway_id,access_control_id,road_surface_condition_id,weather_condition_id,light_condition_id,vehicles_in_accident,fatalities,injuries,tow_away,federal_recordable,state_recordable,cmv_count,truck_count,bus_count,any_hazmat_placard,any_hazmat_released,carrier_count,severity,is_fatal,report_year,report_month,report_month_start,report_dow,report_state_mismatch
0,AK0000000160_20160113,AK0000000160,AK,2016-01-13,0237,2,AK,02,170,02170,True,SUSITNA NORTH,Parks Hwy and Talkeetna Spur Road,Alaska State Troopers,1,3,6,1,2,2,0,0,True,True,False,1,1,0,<NA>,<NA>,1,Tow-away,False,2016,1,2016-01-01,2,False
1,AK0000001724_20170104,AK0000001724,AK,2017-01-04,1649,16,AK,02,170,02170,True,WASILLA,Parks Hwy at Seward Meridian Pkwy,Wasilla City Police,3,3,6,1,2,3,0,5,True,True,False,1,1,0,<NA>,<NA>,1,Injury,False,2017,1,2017-01-01,2,False
2,AK0000002880_20190701,AK0000002880,AK,2019-07-01,1014,10,AK,02,122,02122,True,HOMER,End End Rd and Mariner Dr,Homer City Police,1,<NA>,1,1,1,2,0,0,True,True,False,1,0,1,<NA>,<NA>,1,Tow-away,False,2019,7,2019-07-01,0,False
3,AK0000015164_20150127,AK0000015164,AK,2015-01-27,1349,13,AK,02,170,02170,True,WASILLA,Knik Goose Bay Road and Fern Street,Wasilla City Police,1,2,1,1,1,2,0,0,True,True,False,1,1,0,<NA>,<NA>,0,Tow-away,False,2015,1,2015-01-01,1,False
4,AK0000015321_20150217,AK0000015321,AK,2015-02-17,1459,14,AK,02,170,02170,True,WASILLA,Parks Hwy and 750 ft S from Deskas,Wasilla City Police,1,3,1,1,1,1,0,1,True,True,False,1,1,0,<NA>,<NA>,1,Injury,False,2015,2,2015-02-01,1,False


## Q1. What is one row? (vehicle vs crash)
The raw file has one row per commercial vehicle (CMV) in a crash. `report_number` is shared by every CMV in the same crash.

In [ ]:
raw_2018 = pd.read_parquet(RAW + "crash_2018.parquet")
multi = raw_2018["report_number"].value_counts()
print("2018 raw rows:", len(raw_2018), "| distinct report_numbers:", raw_2018["report_number"].nunique())
example = multi.index[0]
raw_2018.loc[raw_2018["report_number"] == example,
             ["crash_id", "report_number", "report_seq_no", "report_date", "fatalities", "injuries", "dot_number"]]

2018 raw rows: 196568 | distinct report_numbers: 183433


,crash_id,report_number,report_seq_no,report_date,fatalities,injuries,dot_number
39834,3564648,OH0186046896,4,20180313,0,10,2279731
39887,3564645,OH0186046896,1,20180313,0,10,2181673
39907,3564649,OH0186046896,5,20180313,0,10,3093296
40015,3564655,OH0186046896,31,20180313,0,10,239039
40112,3564653,OH0186046896,29,20180313,0,10,383212
40236,3564651,OH0186046896,7,20180313,0,10,499321
40274,3564658,OH0186046896,40,20180313,0,10,121018
40313,3564646,OH0186046896,2,20180313,0,10,191496
40333,3666839,OH0186046896,35,20180313,0,10,None
40342,3564650,OH0186046896,6,20180313,0,10,None


In [ ]:
print("vehicle rows:", len(vehicle), "| crashes:", len(crash))
crash["cmv_count"].value_counts().sort_index().head(10)

vehicle rows: 2168212 | crashes: 2026244


cmv_count
1     1897309
2      119120
3        8023
4        1303
5         243
6         101
7          47
8          22
9          19
10         10
Name: count, dtype: Int64

## Q2. Summing fatalities over vehicle rows over-counts

In [5]:
v = vehicle[["crash_key"]].merge(crash[["crash_key", "fatalities", "injuries"]], on="crash_key")
pd.DataFrame({
    "counted_per_crash": [crash["fatalities"].sum(), crash["injuries"].sum()],
    "summed_over_vehicle_rows": [v["fatalities"].sum(), v["injuries"].sum()],
}, index=["fatalities", "injuries"])

,counted_per_crash,summed_over_vehicle_rows
fatalities,61560,69781
injuries,1057235,1160548


## Q3. Report numbers that are reused across different dates

In [6]:
dates_per_report = crash.groupby("report_number")["report_date"].agg(["nunique", "min", "max"])
reused = dates_per_report[dates_per_report["nunique"] > 1].sort_values("nunique", ascending=False)
print("report_numbers used on >1 date:", len(reused))
reused.head(10)

report_numbers used on >1 date: 6636


,nunique,min,max
report_number,,,
IL0000000000,72,2016-03-14,2026-05-28
MS9030170001,16,2017-01-11,2018-01-02
MS9016240002,14,2024-01-09,2024-12-05
MS9030220001,14,2022-01-09,2024-08-06
MS9038220002,14,2022-01-27,2024-08-06
MS9030230001,14,2023-01-08,2023-12-06
MS9047200002,13,2020-01-27,2020-10-29
MS9038240001,13,2024-02-19,2024-12-06
MS9075190001,12,2019-01-03,2019-12-03


In [7]:
# which states reuse report numbers the most
reused.index.str[:2].value_counts().head(10)

report_number
MS    2362
AZ    1570
PA     751
ME     527
CO     363
AL     175
OK     118
LA     116
CA     110
MD      65
Name: count, dtype: Int64

## Q4. Vehicles in the same crash that disagree on crash-level fields

In [8]:
raw_cols = ["crash_id", "report_number", "report_date", "fatalities", "injuries", "weather_condition_id", "county_code"]
raw = pd.concat([pd.read_parquet(RAW + f"crash_{y}.parquet", columns=raw_cols) for y in range(2015, 2027)])
raw["crash_key"] = raw["report_number"] + "_" + raw["report_date"]

multi_raw = raw[raw.duplicated("crash_key", keep=False)]
disagree = multi_raw.groupby("crash_key")[["fatalities", "injuries", "weather_condition_id", "county_code"]].nunique()
(disagree > 1).sum()

fatalities               58
injuries                406
weather_condition_id     83
county_code             265
dtype: int64

In [9]:
# one example crash where the vehicle rows report different fatality counts
bad_key = disagree[disagree["fatalities"] > 1].index[0]
raw[raw["crash_key"] == bad_key]

,crash_id,report_number,report_date,fatalities,injuries,weather_condition_id,county_code,crash_key
3427,3157247,AL0006603546,20160109,1,0,1,051,AL0006603546_20160109
3433,3157246,AL0006603546,20160109,0,0,1,051,AL0006603546_20160109


## Q5. Crashes by year - is the latest year complete? 2020 Covid Shift

In [10]:
by_year = crash.groupby("report_year").agg(
    crashes=("crash_key", "size"),
    fatalities=("fatalities", "sum"),
    fatal_crashes=("is_fatal", "sum"),
)
by_year["pct_fatal"] = (100 * by_year["fatal_crashes"] / by_year["crashes"]).round(2)
by_year

,crashes,fatalities,fatal_crashes,pct_fatal
report_year,,,,
2015,161312,4728,4202,2.6
2016,167742,4964,4415,2.63
2017,171747,5116,4576,2.66
2018,184115,5254,4677,2.54
2019,182274,5353,4793,2.63
2020,156691,5190,4653,2.97
2021,184499,6036,5369,2.91
2022,185116,6190,5514,2.98
2023,174711,5640,5081,2.91


In [11]:
# monthly counts for the most recent months - the last months are still being filled in
crash[crash["report_date"] >= pd.Timestamp("2025-01-01").date()].groupby("report_month_start").size()

report_month_start
2025-01-01    15205
2025-02-01    13628
2025-03-01    13765
2025-04-01    13208
2025-05-01    12761
2025-06-01    13040
2025-07-01    13549
2025-08-01    13081
2025-09-01    14112
2025-10-01    15032
2025-11-01    13041
2025-12-01    15208
2026-01-01    13513
2026-02-01    11770
2026-03-01    13900
2026-04-01    13082
2026-05-01    12543
2026-06-01    12588
2026-07-01    12639
2026-08-01    12140
2026-09-01    11294
2026-10-01     5012
dtype: int64

In [12]:
# reporting lag: days between the crash and when the record was added to FMCSA's system
lag = vehicle[["crash_key", "add_date", "transaction_code"]].merge(crash[["crash_key", "report_date"]], on="crash_key")
lag["days_to_upload"] = (lag["add_date"] - pd.to_datetime(lag["report_date"])).dt.days
print("share of records changed after upload:", round((lag["transaction_code"] == "C").mean() * 100, 1), "%")
lag["days_to_upload"].describe(percentiles=[0.5, 0.75, 0.9, 0.99])

share of records changed after upload: 59.0 %


count    2.168212e+06
mean     4.287564e+01
std      9.408186e+01
min      0.000000e+00
50%      2.000000e+01
75%      4.000000e+01
90%      8.100000e+01
99%      4.850000e+02
max      3.736000e+03
Name: days_to_upload, dtype: float64

## Q6. Do states report the same way? (fatal share vs tow-away-only share)

In [13]:
by_state = crash.groupby("state").agg(
    crashes=("crash_key", "size"),
    fatalities=("fatalities", "sum"),
    pct_fatal=("is_fatal", "mean"),
    pct_towaway_only=("severity", lambda s: (s == "Tow-away").mean()),
)
by_state[["pct_fatal", "pct_towaway_only"]] = (100 * by_state[["pct_fatal", "pct_towaway_only"]]).round(2)
by_state.sort_values("crashes", ascending=False).head(10)

,crashes,fatalities,pct_fatal,pct_towaway_only
state,,,,
TX,226221,8469,3.3,55.82
CA,149862,5125,3.08,63.22
FL,108882,3984,3.35,59.46
PA,89118,1946,1.97,64.29
IL,86431,1952,2.02,61.71
NC,83921,1903,2.03,53.68
GA,81946,2496,2.74,62.35
OH,77334,2041,2.35,60.26
NY,76166,1574,1.91,43.60


In [14]:
by_state.sort_values("pct_fatal", ascending=False)

,crashes,fatalities,pct_fatal,pct_towaway_only
state,,,,
MP,3,2,66.67,0.00
AS,5,18,40.0,40.00
PR,197,42,20.3,32.49
AK,476,62,11.97,59.66
VI,13,1,7.69,38.46
NM,9923,758,6.7,63.08
NV,8570,496,5.19,58.33
ID,8595,482,4.81,57.48
SD,4699,232,4.38,85.87


## Q7. Crash state vs. the state the carrier is based in

In [15]:
cs = carrier[carrier["dot_number"].notna()][["crash_key", "crash_carrier_state"]].merge(
    crash[["crash_key", "state"]], on="crash_key")
cs["carrier_from_other_state"] = cs["crash_carrier_state"] != cs["state"]
print("share of identified carriers based in another state:", round(100 * cs["carrier_from_other_state"].mean(), 1), "%")
cs.groupby("state")["carrier_from_other_state"].mean().mul(100).round(1).sort_values(ascending=False).head(10)

share of identified carriers based in another state: 51.9 %


state
WY    88.3
NM    80.2
WV    78.9
NV    74.1
MT    74.0
KY    69.0
TN    68.9
VA    67.0
AR    66.8
IN    66.7
Name: carrier_from_other_state, dtype: Float64

## Q8. Vehicles with no USDOT number (can't be joined to carrier registration data)

In [16]:
print("share of vehicle rows with no USDOT number:", round(100 * (~vehicle["has_dot_number"]).mean(), 1), "%")
vehicle.groupby("ci_status_code", dropna=False).agg(
    rows=("crash_id", "size"),
    pct_no_dot=("has_dot_number", lambda s: round(100 * (~s).mean(), 1)),
).sort_values("rows", ascending=False)

share of vehicle rows with no USDOT number: 21.5 %


,rows,pct_no_dot
ci_status_code,,
C,1702349,0.0
X,261749,100.0
I,187635,99.7
P,13952,100.0
N,2276,92.2
<NA>,160,100.0
U,80,100.0
H,11,100.0


In [17]:
no_dot = vehicle[["crash_key", "has_dot_number"]].merge(crash[["crash_key", "state"]], on="crash_key")
no_dot.groupby("state")["has_dot_number"].apply(lambda s: round(100 * (~s).mean(), 1)).sort_values(ascending=False)

state
AS    100.0
MP    100.0
VI    100.0
PR     85.1
HI     55.8
NC     32.3
TX     31.3
LA     30.3
AR     30.0
MO     29.4
OH     29.4
IL     29.0
VA     26.0
FL     25.6
NJ     24.3
SC     22.5
MD     21.8
OK     20.8
TN     20.1
MI     20.1
MS     19.9
ID     19.8
DC     19.5
NY     19.1
ND     18.5
DE     18.0
AL     17.8
IA     17.7
OR     17.4
CA     16.7
MN     16.6
NH     15.8
SD     15.7
MA     15.5
RI     15.4
UT     15.3
NE     15.0
ME     14.9
GA     14.3
KS     13.9
AZ     13.7
WI     13.2
PA     12.7
AK     12.5
CT     12.5
IN     12.2
KY     11.5
WV     10.8
WA     10.8
VT      9.5
CO      8.9
MT      8.3
NV      8.1
GU      7.7
NM      4.9
WY      4.0
Name: has_dot_number, dtype: float64

## Q9. Interstate vs intrastate vs non-motor carriers

In [18]:
carrier["crash_carrier_interstate"].value_counts(dropna=False)

crash_carrier_interstate
Y    1465788
N     443129
X     259295
Name: count, dtype: Int64

## Q10. A few huge carriers among many small ones

In [19]:
per_carrier = carrier[carrier["dot_number"].notna()].groupby("dot_number").agg(
    name=("crash_carrier_name", "first"), vehicles_in_crashes=("crash_id", "size"))
print("carriers:", len(per_carrier))
print("share of carriers with exactly 1 crash vehicle:", round(100 * (per_carrier["vehicles_in_crashes"] == 1).mean(), 1), "%")
big = per_carrier["vehicles_in_crashes"] >= 100
print("share of rows from carriers with 100+:", round(100 * per_carrier.loc[big, "vehicles_in_crashes"].sum() / per_carrier["vehicles_in_crashes"].sum(), 1), "%")
per_carrier.sort_values("vehicles_in_crashes", ascending=False).head(10)

carriers: 452019
share of carriers with exactly 1 crash vehicle: 59.6 %
share of rows from carriers with 100+: 25.2 %


,name,vehicles_in_crashes
dot_number,,
21800,UNITED PARCEL SERVICE INC,14981
265752,FEDEX GROUND PACKAGE SYSTEM INC,11438
80806,J B HUNT,9516
54283,SWIFT TRANSPORTATION CO OF ARIZONA LLC,8515
86876,FEDEX EXPRESS,5929
53467,WERNER ENTERPRISES INC,5666
16130,RYDER TRUCK RENTAL INC,5577
264184,SCHNEIDER NATIONAL CARRIERS INC,5172
239039,FEDEX FRIGHT,4945


## Q11. Hazmat - and how often the placard field is blank

In [20]:
vehicle.groupby("truck_bus_ind").agg(
    rows=("crash_id", "size"),
    pct_placard=("vehicle_hazmat_placard", lambda s: round(100 * s.fillna(False).mean(), 2)),
    pct_released=("hazmat_released", lambda s: round(100 * s.fillna(False).mean(), 2)),
    pct_placard_blank=("vehicle_hazmat_placard", lambda s: round(100 * s.isna().mean(), 1)),
)

,rows,pct_placard,pct_released,pct_placard_blank
truck_bus_ind,,,,
B,182331,0.07,0.06,15.9
T,1985881,1.99,0.48,11.8


## Q12. Vehicle configuration

In [21]:
vehicle.merge(dim_config, left_on="vehicle_configuration_id", right_on="code", how="left")["description"] \
    .value_counts(dropna=False)

description
Tractor/semitrailer                   1002099
Single-unit truck (2 axle, 6 tire)     423804
Single-unit truck (3+ axles)           228546
Truck/trailer                          188452
Bus (>15 seats)                        153681
Truck tractor (bobtail)                 56485
Tractor/double                          33555
Bus (9-15 seats)                        28650
<NA>                                    28516
Unknown heavy truck >10,000 lbs         22143
Tractor/triple                           1712
Light truck (HM placard)                  456
Passenger car (HM placard)                113
Name: count, dtype: Int64

## Q13. Are the crash event codes reliable?
"Other movable object", "explosion or fire" and "train" show up far more often than expected.

In [22]:
event.groupby(["event_id", "event_desc_raw"], dropna=False).size().sort_values(ascending=False).head(20)

event_id  event_desc_raw                                    
20        COLLISION INVOLVING OTHER MOVABLE OBJECT              825441
13        COLLISION INVOLVING MOTOR VEHICLE IN TRANSPORT        680865
18        COLLISION INVOLVING FIXED OBJECT                      280999
6         NONCOLLISION:EXPLOSION OR FIRE                        146298
1         NONCOLLISION:RAN OFF ROAD                             142123
98        OTHER:OTHER                                            99857
          OTHER                                                  96727
17        COLLISION INVOLVING ANIMAL                             86693
3         NONCOLLISION:OVERTURN (ROLLOVER)                       67722
8         NONCOLLISION:CROSS MEDIAN/CENTERLINE                   40112
11        NONCOLLISION:UNKNOWN                                   34192
5         NONCOLLISION:CARGO LOSS OR SHIFT                       25494
14        COLLISION INVOLVING PARKED MOTOR VEHICLE               22894
15        COLLIS

In [23]:
# is it one state causing it? (no - it is spread across states)
ev_state = event.merge(crash[["crash_key", "state"]], on="crash_key")
for eid in [20, 6, 15]:
    print(dim_event.loc[dim_event["code"] == eid, "description"].iloc[0])
    print(ev_state[ev_state["event_id"] == eid]["state"].value_counts(normalize=True).mul(100).round(1).head(5), "\n")

Collision: other movable object
state
CA    7.4
FL    7.4
GA    5.8
PA    5.3
IL    5.1
Name: proportion, dtype: Float64 

Non-collision: explosion or fire
state
MO    7.9
OH    7.0
NC    6.3
VA    5.0
LA    5.0
Name: proportion, dtype: Float64 

Collision: train


state
WY    6.6
MI    5.7
OH    5.1
IL    4.3
PA    4.0
Name: proportion, dtype: Float64 



## Q14. Light, time of day and day of week vs. fatal share

In [24]:
crash.merge(dim_light, left_on="light_condition_id", right_on="code", how="left").groupby("description", dropna=False) \
    .agg(crashes=("crash_key", "size"), pct_fatal=("is_fatal", "mean")) \
    .assign(pct_fatal=lambda d: (100 * d["pct_fatal"]).round(2)).sort_values("crashes", ascending=False)

,crashes,pct_fatal
description,,
Daylight,1458490,2.25
Dark - not lighted,263626,4.75
Dark - lighted,213803,3.15
Dawn,47908,3.19
Dusk,26816,3.2
Dark - unknown roadway lighting,8784,3.03
Unknown or blank,3310,2.6
<NA>,1792,8.04
Other,1715,4.43


In [25]:
crash.groupby("report_dow").agg(crashes=("crash_key", "size"), pct_fatal=("is_fatal", "mean")) \
    .assign(pct_fatal=lambda d: (100 * d["pct_fatal"]).round(2)).rename(index=dict(enumerate(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])))

,crashes,pct_fatal
report_dow,,
Mon,340010,2.61
Tue,361373,2.51
Wed,357252,2.59
Thu,353053,2.61
Fri,343887,2.66
Sat,159292,3.42
Sun,111377,3.64


In [26]:
crash.groupby("report_hour").agg(crashes=("crash_key", "size"), pct_fatal=("is_fatal", "mean")) \
    .assign(pct_fatal=lambda d: (100 * d["pct_fatal"]).round(2))

,crashes,pct_fatal
report_hour,,
0,42009,4.02
1,31083,4.83
2,32752,5.27
3,33454,4.91
4,41364,4.07
5,62850,3.78
6,99292,3.0
7,134194,2.16
8,137895,1.91


## Q15. Bad weather has a *lower* fatal share

In [27]:
crash.merge(dim_weather, left_on="weather_condition_id", right_on="code", how="left").groupby("description", dropna=False) \
    .agg(crashes=("crash_key", "size"), pct_fatal=("is_fatal", "mean")) \
    .assign(pct_fatal=lambda d: (100 * d["pct_fatal"]).round(2)).sort_values("crashes", ascending=False)

,crashes,pct_fatal
description,,
No adverse condition,1596080,2.84
Rain,190931,2.0
Other,70001,2.92
Snow,67262,1.16
Unknown or blank,29639,2.64
<NA>,25595,3.66
Fog,17805,4.78
"Sleet, hail",12060,1.6
"Blowing sand, soil, dirt, or snow",9609,2.05


## Q16. How complete is each condition / location field?

In [28]:
cols = ["weather_condition_id", "light_condition_id", "road_surface_condition_id", "access_control_id",
        "trafficway_id", "report_hour", "county_fips", "city"]
completeness = pd.DataFrame({"pct_missing": (100 * crash[cols].isna().mean()).round(1)})
completeness.loc["county_fips (not a real county)", "pct_missing"] = round(100 * (~crash["county_fips_valid"]).mean(), 1)
completeness

,pct_missing
weather_condition_id,1.3
light_condition_id,0.1
road_surface_condition_id,0.3
access_control_id,41.4
trafficway_id,7.8
report_hour,0.0
county_fips,0.0
city,12.6
county_fips (not a real county),1.0


## Q17. Federally recordable vs state recordable

In [29]:
crash.groupby(["federal_recordable", "state_recordable"], dropna=False).size()

federal_recordable  state_recordable
False               False                   196
                    True                   1428
True                False                201282
                    True                1780028
                    <NA>                  43310
dtype: int64